# 🦋 Insect Anatomy YOLO-Seg (v2: High-Res + Data Augmentation)

This notebook trains a higher-capacity YOLO instance segmentation model on your insect annotations.

### What's New in v2:
1. **Bigger Model (`yolov8s-seg` / `yolo11s-seg`):** ~11M parameters (3.5x capacity of nano) to capture complex wing veins, pectinate antennae, and legs on textured backgrounds.
2. **Higher Resolution (`imgsz=1024`):** Retains delicate sub-millimeter appendages that get lost at 640px.
3. **Domain Adaptation Augmentations:** Aggressive HSV lighting jitter (sunlight vs shade vs flash), scale variations (+/- 50% for macro vs full insect), random rotation (+/- 30°), and Mosaic/Mixup.
4. **Verified iNat Samples Mixed In:** Learns directly from real iNaturalist outdoor photos.

### 1. Check Hardware & Sync Latest Annotations

In [ ]:
import torch
import os

device = 0 if torch.cuda.is_available() else 'cpu'
print(f"🚀 PyTorch version: {torch.__version__}")
print(f"⚡ Compute device:  {device} ({'GPU: ' + torch.cuda.get_device_name(0) if device != 'cpu' else 'CPU multi-core'})")

# Sync any recently verified iNat samples into the training dataset
!python cluster_yolo_training/update_dataset.py

### 2. Verify Dataset
Checks images and classes in `dataset.yaml`.

In [ ]:
import glob
import yaml

yaml_path = 'dataset/dataset.yaml' if os.path.exists('dataset/dataset.yaml') else 'cluster_yolo_training/dataset/dataset.yaml'
with open(yaml_path) as f:
    dataset_cfg = yaml.safe_load(f)

print("Classes:", dataset_cfg.get('names'))
train_imgs = glob.glob('dataset/images/train/*.jpg') or glob.glob('cluster_yolo_training/dataset/images/train/*.jpg')
val_imgs = glob.glob('dataset/images/val/*.jpg') or glob.glob('cluster_yolo_training/dataset/images/val/*.jpg')
print(f"Train samples: {len(train_imgs)} (includes field + iNat!)")
print(f"Val samples:   {len(val_imgs)}")

### 3. Start Training (v2 with High-Res & Augmentations)
- Model: `yolov8s-seg.pt` (Small)
- Resolution: `1024` (preserves fine antenna and leg textures)
- Epochs: `150` with Cosine Annealing learning rate

In [ ]:
from ultralytics import YOLO

# Load small model with 3.5x capacity of nano
model = YOLO('yolov8s-seg.pt')

batch_size = 16 if torch.cuda.is_available() else 4
epochs = 150

results = model.train(
    data=yaml_path,
    epochs=epochs,
    batch=batch_size,
    imgsz=640,
    device=device,
    patience=30,
    cos_lr=True,            # Cosine learning rate schedule
    # --- Domain Augmentations for iNat & Macro Lighting ---
    hsv_h=0.015,            # Hue jitter
    hsv_s=0.7,              # Saturation jitter (sun vs shade)
    hsv_v=0.4,              # Value / brightness jitter (shadows vs flash)
    degrees=30.0,           # Random rotations up to 30 deg
    scale=0.5,              # Multi-scale zoom (+/- 50%)
    fliplr=0.5,             # Horizontal flips
    flipud=0.5,             # Vertical flips
    mosaic=1.0,             # Multi-image context mosaic
    mixup=0.15,             # Blend overlapping structures
    close_mosaic=15,        # Turn off mosaic for clean boundary learning
    project='runs',
    name='insect_anatomy_v2',
    exist_ok=True,
    save=True,
    plots=True
)

### 4. Inspect Results & Loss Curves

In [ ]:
from IPython.display import Image, display

results_png = 'runs/insect_anatomy_v2/results.png'
if os.path.exists(results_png):
    display(Image(results_png))
else:
    print("Results plot not found yet.")

### 4b. Visual Validation Predictions
Run inference on the validation images and view predicted masks overlaid.

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import glob, os

# 1. Load trained model
best_weights = 'cluster_yolo_training/weights/best.pt' if os.path.exists('cluster_yolo_training/weights/best.pt') else 'weights/best.pt'
val_model = YOLO(best_weights)

# 2. Predict on validation set (max_det=30 prevents NMS timeout)
val_dir = 'dataset/images/val' if os.path.exists('dataset/images/val') else 'cluster_yolo_training/dataset/images/val'
val_results = val_model.predict(
    source=val_dir,
    conf=0.25,
    iou=0.45,
    max_det=30,
    save=True,
    name='val_visuals',
    exist_ok=True
)

# 3. Get exact output directory directly from YOLO
output_dir = val_results[0].save_dir
pred_imgs = sorted(glob.glob(os.path.join(output_dir, '*.jpg')))
print(f"✔ Found {len(pred_imgs)} images in: {output_dir}\n")
for p in pred_imgs[:5]:
    print(f"📷 {os.path.basename(p)}")
    display(Image(p, width=500))

### 5. Save & Export Weights

In [ ]:
import shutil

os.makedirs('cluster_yolo_training/weights', exist_ok=True)
best_src = 'runs/insect_anatomy_v2/weights/best.pt'
best_dst = 'cluster_yolo_training/weights/best.pt'

if os.path.exists(best_src):
    shutil.copy2(best_src, best_dst)
    print(f"✔ Saved best model to: {best_dst}")
    
    best_model = YOLO(best_dst)
    onnx_path = best_model.export(format='onnx', imgsz=1024)
    print(f"✔ Exported to ONNX: {onnx_path}")

### 6. Re-predict Remaining Active Backlog (AI-assisted Pre-annotation)
Runs the improved model to generate sharper masks on the remaining images.

In [ ]:
!python cluster_yolo_training/predict_and_annotate.py --weights cluster_yolo_training/weights/best.pt --target data/active_annotation_set --overwrite-ai